# Chain Rule and Backpropagation

Companion notebook for the [Chain Rule and Backpropagation](https://ml-viz-ruby.vercel.app/courses/calculus-for-ml/02-chain-rule-and-backpropagation) lesson.

We implement backpropagation from scratch and compare to numerical gradients.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

## Intuition — backprop is the chain rule, organized

The **chain rule** says the derivative of a composition is the *product* of the local
derivatives along the way: if `L` depends on `h`, which depends on `g`, which depends on
`x`, then `dL/dx = dL/dh · dh/dg · dg/dx`. **Backpropagation** is nothing more than this
rule applied to a whole network's computational graph — swept **right-to-left** so each
node multiplies the gradient flowing into it (*upstream*) by its own *local* derivative
and passes the result back. Doing it right-to-left reuses shared sub-results, which is
what makes training deep networks affordable. We'll build it by hand, then watch
autodiff do the identical thing automatically.

## 1. From scratch — one node at a time

Start with a scalar composition `h = σ(3x²)`. The **forward pass** builds the graph
`x → g = 3x² → h = σ(g)`, caching each intermediate. The **backward pass** starts from
`dL/dh = 1` and multiplies by one local derivative per step: `σ'(g)` to get `dL/dg`, then
`6x` (the derivative of `3x²`) to get `dL/dx`. A finite-difference check confirms it.

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))
def sigmoid_grad(z): return sigmoid(z) * (1 - sigmoid(z))

x = 0.5   # input

# Forward pass (build computational graph)
g = 3 * x**2        # g = 3x²
h = sigmoid(g)      # h = σ(g)

print('=== Forward Pass ===')
print(f'x = {x}')
print(f'g = 3x² = {g}')
print(f'h = σ(g) = {h:.6f}')

# Backward pass
dL_dh = 1.0             # upstream gradient
dL_dg = dL_dh * sigmoid_grad(g)  # local gradient of σ
dL_dx = dL_dg * 6 * x            # local gradient of 3x²

print('\n=== Backward Pass ===')
print(f'dL/dh = {dL_dh}')
print(f'dL/dg = dL/dh × σ\'(g) = {dL_dh:.4f} × {sigmoid_grad(g):.4f} = {dL_dg:.6f}')
print(f'dL/dx = dL/dg × 6x   = {dL_dg:.6f} × {6*x:.4f} = {dL_dx:.6f}')

# Verify numerically
eps = 1e-7
h_plus  = sigmoid(3*(x+eps)**2)
h_minus = sigmoid(3*(x-eps)**2)
numerical = (h_plus - h_minus) / (2 * eps)
print(f'\nNumerical dh/dx = {numerical:.6f}  (matches: {abs(dL_dx - numerical) < 1e-5})')

**What to notice:** `dL/dx` from the two-step backward pass matches the numerical
`dh/dx` — the chain rule is just "multiply the local slopes." Each arrow in the graph
contributed exactly one factor. That's the whole mechanism; the rest is bookkeeping.

## Manual backprop through a 1-hidden-layer network

Same recipe, scaled up: a $3 \to 4$ (ReLU) $\to 1$ (sigmoid) network with binary
cross-entropy loss. We do the forward pass, run backprop **by hand** (one line per
node, upstream x local), then verify *every* gradient against a central-difference
numerical gradient.

In [ ]:
def relu(z): return np.maximum(0, z)
def relu_grad(z): return (np.asarray(z) > 0).astype(float)  # scalar-safe

rng = np.random.default_rng(0)

# Network: 3 inputs -> 4 hidden (ReLU) -> 1 output (sigmoid)
x  = np.array([1.0, 2.0, 3.0])
W1 = rng.standard_normal((4, 3)) * 0.1
b1 = np.zeros(4)
w2 = rng.standard_normal(4) * 0.1
b2 = 0.0
y  = 1.0  # true label


def forward(W1, b1, w2, b2):
    """Forward pass returning the scalar loss (used for both training and FD checks)."""
    z1    = W1 @ x + b1
    a1    = relu(z1)
    z2    = w2 @ a1 + b2
    y_hat = sigmoid(z2)
    L     = -(y * np.log(y_hat + 1e-12) + (1 - y) * np.log(1 - y_hat + 1e-12))
    return L, (z1, a1, z2, y_hat)


L, (z1, a1, z2, y_hat) = forward(W1, b1, w2, b2)

# Backward pass: one line per node, upstream x local
dL_dz2 = y_hat - y                 # BCE + sigmoid collapse to (y_hat - y)
dL_dw2 = dL_dz2 * a1               # x a1
dL_db2 = dL_dz2                     # x 1
dL_da1 = dL_dz2 * w2               # x w2
dL_dz1 = dL_da1 * relu_grad(z1)    # x ReLU'(z1)
dL_dW1 = np.outer(dL_dz1, x)       # x x^T
dL_db1 = dL_dz1                     # x 1

print('Loss: {0:.6f}'.format(L))
print('Prediction: {0:.4f}'.format(y_hat))
print('dL/dw2 (shape {0}): {1}'.format(dL_dw2.shape, dL_dw2.round(6)))
print('dL/dW1 (shape {0}):\n{1}'.format(dL_dW1.shape, dL_dW1.round(6)))


# --- Finite-difference check of every analytic gradient ---
def numerical_grad(param_name, base):
    """Central-difference gradient of L w.r.t. a copy of one parameter array/scalar."""
    eps = 1e-6
    arr = np.atleast_1d(np.array(base, dtype=float))
    grad = np.zeros_like(arr)
    for idx in np.ndindex(arr.shape):
        plus = arr.copy();  plus[idx]  += eps
        minus = arr.copy(); minus[idx] -= eps
        kwargs_p = {'W1': W1, 'b1': b1, 'w2': w2, 'b2': b2}
        kwargs_m = dict(kwargs_p)
        kwargs_p[param_name] = plus.reshape(np.shape(base))
        kwargs_m[param_name] = minus.reshape(np.shape(base))
        Lp, _ = forward(**kwargs_p)
        Lm, _ = forward(**kwargs_m)
        grad[idx] = (Lp - Lm) / (2 * eps)
    return grad.reshape(np.shape(base))


checks = {
    'W1': (dL_dW1, numerical_grad('W1', W1)),
    'b1': (dL_db1, numerical_grad('b1', b1)),
    'w2': (dL_dw2, numerical_grad('w2', w2)),
    'b2': (np.array(dL_db2), numerical_grad('b2', b2)),
}

print('\n=== Gradient check (analytic vs numerical) ===')
for name, (analytic, numeric) in checks.items():
    max_err = np.max(np.abs(np.asarray(analytic) - np.asarray(numeric)))
    print('{0:>3}: max abs error = {1:.2e}  (ok: {2})'.format(name, max_err, max_err < 1e-5))

**What to notice:** every hand-derived gradient (`W1`, `b1`, `w2`, `b2`) matches its
central-difference estimate to ~`1e-6`. Two things to internalize: **shapes work out**
(`dL/dW1` is an outer product `dL_dz1 ⊗ x`, same shape as `W1`), and the sigmoid+BCE
pair collapses to the clean `dL/dz2 = ŷ − y`. Backprop for any network is this same
loop, one line per node.

## 2. The library way — autodiff reproduces backprop

Frameworks run exactly this backward pass for you. The cell rebuilds the same
3→4→1 network in `jax`, then `jax.grad` differentiates the loss with respect to the
parameter tuple and returns gradients for `W1, b1, w2, b2` at once — which we assert
equal the hand-derived ones. `loss.backward()` in PyTorch is the same operation.

In [ ]:
import jax, jax.numpy as jnp

def loss_jax(params, x, y):
    W1, b1, w2, b2 = params
    z1 = W1 @ x + b1
    a1 = jnp.maximum(0.0, z1)                       # ReLU
    z2 = w2 @ a1 + b2
    y_hat = 1.0 / (1.0 + jnp.exp(-z2))              # sigmoid
    return -(y * jnp.log(y_hat + 1e-12) + (1 - y) * jnp.log(1 - y_hat + 1e-12))

params = (jnp.array(W1), jnp.array(b1), jnp.array(w2), jnp.array(b2))
gW1, gb1, gw2, gb2 = jax.grad(loss_jax)(params, jnp.array(x), y)

print('autodiff dL/dw2 :', np.array(gw2).round(6))
print('by-hand  dL/dw2 :', dL_dw2.round(6))
assert np.allclose(np.array(gW1), dL_dW1, atol=1e-5), "autodiff must match hand backprop (W1)"
assert np.allclose(np.array(gw2), dL_dw2, atol=1e-5), "autodiff must match hand backprop (w2)"
print('\nautodiff gradients == hand backprop for every parameter ✓')

**What to notice:** the autodiff gradients are identical to our by-hand backprop, to
numerical precision. Autodiff isn't a different algorithm — it *is* backprop, traced
automatically through whatever operations you wrote. Understanding the hand version is
understanding what the framework does under `.backward()`.

## Vanishing gradients: sigmoid vs ReLU

In [ ]:
def simulate_gradient_flow(activation, activation_grad, n_layers=20, x0=1.0):
    """Simulate gradient magnitude through n_layers with a given activation."""
    rng = np.random.default_rng(42)
    gradient = 1.0
    grad_magnitudes = [gradient]

    for _ in range(n_layers):
        z = rng.standard_normal() * 0.5   # random pre-activation
        local_grad = activation_grad(z)
        gradient *= local_grad
        grad_magnitudes.append(abs(gradient))

    return grad_magnitudes

sigmoid_grads = simulate_gradient_flow(sigmoid, sigmoid_grad, n_layers=20)
relu_grads    = simulate_gradient_flow(relu, relu_grad, n_layers=20)

fig, ax = plt.subplots(figsize=(10, 5))
layers = range(len(sigmoid_grads))
ax.semilogy(layers, sigmoid_grads, 'o-', color='#f97316', lw=2, ms=6, label='Sigmoid')
ax.semilogy(layers, relu_grads,    's-', color='#6366f1', lw=2, ms=6, label='ReLU')
ax.set_xlabel('Layer (counting backward)'); ax.set_ylabel('|gradient| (log scale)')
ax.set_title('Gradient magnitude through 20 layers: Sigmoid vs ReLU', pad=12)
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

print(f'Sigmoid gradient after 20 layers: {sigmoid_grads[-1]:.2e}')
print(f'ReLU    gradient after 20 layers: {relu_grads[-1]:.2e}')

**What to notice:** the sigmoid path decays smoothly toward ~`1e-13` — every layer
multiplies by ≤ 0.25, so the product shrinks *no matter what*. ReLU's derivative is
instead either **1** (active) or **0** (inactive): along this single random path it
passes the gradient **undamped** until the first negative pre-activation zeroes it out
(a *dead* path → `0`). That's the key contrast — sigmoid guarantees exponential
shrinkage everywhere, while ReLU neither shrinks nor grows a gradient where it's active,
so across the many parallel paths of a real network the signal keeps flowing.

## 4. Gotchas & failure modes

- **`log(0)` blows up.** BCE takes `log(ŷ)`; a confidently-wrong prediction pushes `ŷ`
  to 0 or 1, sending the loss to `∞` and gradients to `nan`. The `+1e-12` epsilon (or a
  fused `sigmoid_cross_entropy`) prevents it.
- **Dead ReLU.** When a unit's pre-activation is negative, `ReLU'(z) = 0`, so *no*
  gradient flows through it — if it's stuck there for every input, it never recovers.
- **Vanishing vs exploding.** Multiplying many local gradients `< 1` underflows to 0;
  many `> 1` explodes. Both break deep training (fixes: ReLU, normalization, residuals,
  gradient clipping).
- **Memory cost.** Autodiff must **cache the forward activations** to compute the
  backward pass — which is why training uses far more memory than inference.

In [ ]:
# log(0) instability and the epsilon fix
y_hat = 0.0                                  # a confidently wrong probability
print('log(0)        =', -np.inf, '-> loss = inf, gradients = nan')
print('log(0 + 1e-12) =', np.log(y_hat + 1e-12), '-> finite')

# vanishing vs exploding: a product of local gradients through 20 layers
print('\n20 factors of 0.25 (sigmoid tail):', np.prod([0.25]*20), '-> vanishes')
print('20 factors of 1.5  (large weights):', np.prod([1.5]*20),  '-> explodes')

**What to notice:** `log(0)` is `−∞` but `log(1e-12)` is a large *finite* number, so the
epsilon keeps the loss and its gradient usable. And the two products — `9e-13` vs `3325`
— are vanishing and exploding gradients in miniature: the same multiplication, just with
factors below vs above 1.

## Key takeaways

- **Backprop = chain rule on the computational graph**, swept right-to-left: each node
  multiplies the upstream gradient by its local derivative.
- Do it once by hand and the pattern is clear — **one line per node**, shapes matching
  each parameter, and `sigmoid + BCE → ŷ − y`.
- **Autodiff** (`jax.grad`, `loss.backward()`) is this exact procedure, traced
  automatically — verify it matches by finite differences when in doubt.
- The failure modes are all about the *product* of local gradients: **vanishing**
  (sigmoid depth), **exploding** (large weights), **dead ReLUs**, and **`log(0)`**.

**Next:** [Multivariable Optimization](https://ml-viz-ruby.vercel.app/courses/calculus-for-ml/03-multivariable-optimization)
— using these gradients to actually navigate a loss surface.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Chain rule by hand

Take the single-neuron function $f(w) = \sigma(wx + b)$ and differentiate it **with respect to the weight** $w$. Two links in the chain:

$$\frac{\partial f}{\partial w} = \underbrace{\sigma'(z)}_{\text{outer}} \cdot \underbrace{\frac{\partial z}{\partial w}}_{= \, x}
\qquad \text{with } z = wx + b, \;\; \sigma'(z) = \sigma(z)\,(1 - \sigma(z))$$

The checks compare your formula against a numerical gradient — the same gradient-checking trick used to debug real backprop code.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def grad_w(x, w, b):
    """d sigmoid(w*x + b) / dw via the chain rule."""
    # TODO(you): the pre-activation z
    z = ...

    # TODO(you): sigma(z), then sigma'(z) = s * (1 - s)
    s = ...

    # TODO(you): chain rule: sigma'(z) times dz/dw (which is x)
    return ...

In [ ]:
# Checks — run me
x, w, b = 2.0, 0.5, -1.0
h = 1e-6
numeric = (sigmoid((w + h) * x + b) - sigmoid((w - h) * x + b)) / (2 * h)
assert abs(grad_w(x, w, b) - numeric) < 1e-8, "chain rule must match the numerical gradient"
assert abs(grad_w(0.0, w, b)) < 1e-12, "x = 0 -> z doesn't depend on w -> zero gradient"
assert abs(grad_w(1.0, 0.0, 0.0) - 0.25) < 1e-12, "sigma'(0) = 0.25, times x = 1"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def grad_w(x, w, b):
    z = w * x + b
    s = sigmoid(z)
    return s * (1 - s) * x
```

</details>

### Exercise 2 — Backprop through a tiny network

A two-weight scalar network: $y = w_2 \cdot \text{ReLU}(w_1 x)$, loss $L = \tfrac{1}{2}(y - t)^2$. Walk the gradient backwards node by node:

$$\frac{\partial L}{\partial y} = y - t, \qquad
\frac{\partial L}{\partial w_2} = (y - t)\,a, \qquad
\frac{\partial L}{\partial w_1} = (y - t)\,w_2 \cdot \mathbb{1}[z > 0] \cdot x$$

where $z = w_1 x$ and $a = \text{ReLU}(z)$. The last two checks make the **dead ReLU** point: when $z < 0$, the gate is shut and *neither* weight gets a gradient.

In [ ]:
def backprop(x, w1, w2, t):
    """Return (dL/dw1, dL/dw2) for y = w2 * relu(w1 * x), L = 0.5 * (y - t)^2."""
    # Forward pass
    z = w1 * x
    a = max(z, 0.0)          # ReLU
    y = w2 * a

    # Backward pass
    # TODO(you): dL/dy
    dy = ...

    # TODO(you): dL/dw2 = dL/dy * dy/dw2 (= a)
    dw2 = ...

    # TODO(you): dL/dw1 = dL/dy * w2 * relu'(z) * x, where relu'(z) is 1 if z > 0 else 0
    dw1 = ...

    return dw1, dw2

In [ ]:
# Checks — run me
def loss(x, w1, w2, t):
    return 0.5 * (w2 * max(w1 * x, 0.0) - t) ** 2

x, w1, w2, t = 1.5, 0.8, -1.2, 1.0
dw1, dw2 = backprop(x, w1, w2, t)
h = 1e-6
num1 = (loss(x, w1 + h, w2, t) - loss(x, w1 - h, w2, t)) / (2 * h)
num2 = (loss(x, w1, w2 + h, t) - loss(x, w1, w2 - h, t)) / (2 * h)
assert abs(dw1 - num1) < 1e-6, "dL/dw1 must match the numerical gradient"
assert abs(dw2 - num2) < 1e-6, "dL/dw2 must match the numerical gradient"

dw1_dead, dw2_dead = backprop(-1.5, 0.8, -1.2, 1.0)
assert dw1_dead == 0.0, "ReLU is dead (z < 0) -> gradient to w1 is blocked"
assert dw2_dead == 0.0, "dead ReLU -> a = 0 -> w2 gets no gradient either"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def backprop(x, w1, w2, t):
    z = w1 * x
    a = max(z, 0.0)
    y = w2 * a
    dy = y - t
    dw2 = dy * a
    dw1 = dy * w2 * (x if z > 0 else 0.0)
    return dw1, dw2
```

</details>